# Лабораторная работа №4: Случайный лес

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Обоснование выбора метрик

## Для классификации (Titanic – предсказание выживания)

Задача бинарной классификации с несбалансированными классами (выживших меньше, чем погибших). Поэтому одной accuracy недостаточно.

**Accuracy (доля правильных ответов)** – базовый показатель, интуитивно понятен. Однако при дисбалансе классов высокая accuracy может достигаться за счёт предсказания только мажоритарного класса. Поэтому используем дополнительные метрики.

**Precision (точность)** – доля верно предсказанных выживших среди всех предсказанных выживших. Показывает, насколько можно доверять положительному прогнозу модели. В контексте Titanic высокая точность означает, что если модель говорит "выживет", то это с большой вероятностью правда.

**Recall (полнота)** – доля выживших, правильно обнаруженных моделью. Показывает, какую часть реально выживших модель смогла найти. Низкий recall означает, что модель пропускает много выживших, что может быть критично, если цель – не упустить потенциально выживших.

**F1-score** – гармоническое среднее precision и recall. Полезна, когда нужно учесть оба аспекта. Особенно важна при дисбалансе классов, так как усредняет precision и recall, не позволяя одному из них быть слишком низким.

## Для регрессии (House Prices – прогноз стоимости жилья)

Задача регрессии, где ошибка измеряется в долларах. Важно понимать величину ошибки и её распределение.

**MAE (Mean Absolute Error)** – средняя абсолютная ошибка. Измеряется в тех же единицах, что и целевая переменная (доллары). Показывает, на сколько в среднем модель ошибается. Устойчива к выбросам, даёт представление о типичной ошибке.

**MSE (Mean Squared Error)** – средняя квадратичная ошибка. Штрафует большие ошибки сильнее из-за возведения в квадрат. Используется как функция потерь во многих алгоритмах. Большая MSE указывает на наличие значительных выбросов в ошибках.

**RMSE (Root Mean Squared Error)** – корень из MSE. Возвращает ошибку в исходных единицах (долларах), сохраняя свойство MSE чувствительности к большим ошибкам, но в более интерпретируемом масштабе. Позволяет сравнивать с MAE: если RMSE значительно больше MAE, значит, есть выбросы с большими ошибками.

Таким боразом, все три метрики дополняют друг друга и дают полную картину качества регрессионной модели.

In [3]:
titanic = pd.read_csv('titanic_train.csv')

titanic_simple = titanic[['Pclass', 'Sex', 'Age', 'Fare', 'Survived']].dropna()
titanic_simple['Sex'] = titanic_simple['Sex'].map({'male': 0, 'female': 1})

X_t = titanic_simple.drop('Survived', axis=1)
y_t = titanic_simple['Survived']

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(X_t, y_t, test_size=0.2, random_state=42)

print("Titanic:")
print("Размер обучающей выборки:", X_train_t.shape)
print("Размер тестовой выборки:", X_test_t.shape)
print("Распределение классов в обучающей выборке:")
print(y_train_t.value_counts())

Titanic:
Размер обучающей выборки: (571, 4)
Размер тестовой выборки: (143, 4)
Распределение классов в обучающей выборке:
Survived
0    337
1    234
Name: count, dtype: int64


In [4]:
house = pd.read_csv('house_train.csv')

house_simple = house[['GrLivArea', 'BedroomAbvGr', 'FullBath', 'SalePrice']].dropna()
X_h = house_simple.drop('SalePrice', axis=1)
y_h = house_simple['SalePrice']

X_train_h, X_test_h, y_train_h, y_test_h = train_test_split(X_h, y_h, test_size=0.2, random_state=42)

print("House Prices:")
print("Размер обучающей выборки:", X_train_h.shape)
print("Размер тестовой выборки:", X_test_h.shape)
print("Диапазон цен в обучающей выборке:")
print(f"Min: {y_train_h.min():.0f}, Max: {y_train_h.max():.0f}, Mean: {y_train_h.mean():.0f}")

House Prices:
Размер обучающей выборки: (1168, 3)
Размер тестовой выборки: (292, 3)
Диапазон цен в обучающей выборке:
Min: 34900, Max: 745000, Mean: 181442


## Обучение моделей и оценка качества

In [5]:
rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train_t, y_train_t)
y_pred_t = rf_clf.predict(X_test_t)

acc = accuracy_score(y_test_t, y_pred_t)
prec = precision_score(y_test_t, y_pred_t)
rec = recall_score(y_test_t, y_pred_t)
f1 = f1_score(y_test_t, y_pred_t)

print("RandomForestClassifier:")
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1-score:  {f1:.4f}")

print('\nДля сравнения метрики базового KNN из лабораторной №1:\nAccuracy:  0.7762\nPrecision: 0.7069\nRecall:    0.7321\nF1-score:  0.7193')

RandomForestClassifier:
Accuracy:  0.7622
Precision: 0.6964
Recall:    0.6964
F1-score:  0.6964

Для сравнения метрики базового KNN из лабораторной №1:
Accuracy:  0.7762
Precision: 0.7069
Recall:    0.7321
F1-score:  0.7193


In [6]:
rf_reg = RandomForestRegressor(random_state=42)
rf_reg.fit(X_train_h, y_train_h)
y_pred_h = rf_reg.predict(X_test_h)

mae = mean_absolute_error(y_test_h, y_pred_h)
mse = mean_squared_error(y_test_h, y_pred_h)
rmse = np.sqrt(mse)

print("RandomForestRegressor:")
print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")

print('\nДля сравнения метрики базового KNN из лабораторной №1:\nMAE:  33713.40\nMSE:  2593685769.64\nRMSE: 50928.24')

RandomForestRegressor:
MAE:  34115.03
MSE:  2423235371.17
RMSE: 49226.37

Для сравнения метрики базового KNN из лабораторной №1:
MAE:  33713.40
MSE:  2593685769.64
RMSE: 50928.24


## Вывод
Для классификации метрики KNN из первой лабораторной работы дают более точные результаты, в то время как метрики KNN для регресии уступают случайному лесу.